# Notebook 04 — Variable Reduction

This notebook compares subsets of the eight predictor blocks on the model-building sample established in Notebook 03. Backward elimination is shown as a stepwise screening path, followed by an exhaustive search of all 255 nonempty block subsets.

No single subset is globally optimal without first specifying the selection criterion. The exhaustive search identifies the best model within this eight-block candidate family for each reported criterion. Model diagnostics and the reserved validation sample are used later to judge the practical choice.

## 1. Import & Reconstruct Model-Building Sample

In [18]:
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd

import statsmodels.formula.api as smf

In [19]:
PROJECT_DIR = Path.cwd().resolve()

if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
OUTPUT_DIR = PROJECT_DIR / "outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_FILE = (
    PROCESSED_DATA_DIR
    / "db1c_ticket_prepared_202606.parquet"
)

df = pd.read_parquet(DATA_FILE)

print(f"Rows:    {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Rows:    90,086
Columns: 16


### 1.1 Reproduce the 50/50 Model-Building Split

In [20]:
SPLIT_SEED = 42
TRAIN_SHARE = 0.50

rng = np.random.default_rng(SPLIT_SEED)

random_order = rng.permutation(len(df))
split_point = int(TRAIN_SHARE * len(df))

train_idx = random_order[:split_point]
validation_idx = random_order[split_point:]

model_df = (
    df.iloc[train_idx]
    .copy()
    .reset_index(drop=True)
)

validation_df = (
    df.iloc[validation_idx]
    .copy()
    .reset_index(drop=True)
)

print(f"Model-building sample: {len(model_df):,}")
print(f"Validation sample:     {len(validation_df):,}")

Model-building sample: 45,043
Validation sample:     45,043


### 1.2 Reproduce Categorical Grouping

In [21]:
MIN_CATEGORY_COUNT = 100

# Carrier grouping based only on model-building data
carrier_counts = model_df["RpCarrier"].value_counts()

major_carriers = carrier_counts[
    carrier_counts >= MIN_CATEGORY_COUNT
].index

model_df["RpCarrier_Model"] = model_df["RpCarrier"].where(
    model_df["RpCarrier"].isin(major_carriers),
    "Other"
)

validation_df["RpCarrier_Model"] = validation_df["RpCarrier"].where(
    validation_df["RpCarrier"].isin(major_carriers),
    "Other"
)

# Origin-state grouping based only on model-building data
state_counts = model_df["OriginState"].value_counts()

major_states = state_counts[
    state_counts >= MIN_CATEGORY_COUNT
].index

model_df["OriginState_Model"] = model_df["OriginState"].where(
    model_df["OriginState"].isin(major_states),
    "Other"
)

validation_df["OriginState_Model"] = validation_df["OriginState"].where(
    validation_df["OriginState"].isin(major_states),
    "Other"
)

print(
    f"Carrier model categories: "
    f"{model_df['RpCarrier_Model'].nunique()}"
)

print(
    f"Origin-state model categories: "
    f"{model_df['OriginState_Model'].nunique()}"
)

Carrier model categories: 12
Origin-state model categories: 43


#### Interpretation

Notebook 04 reproduces exactly the same model-building sample and categorical grouping rules used in Notebook 03.

The validation sample remains reserved and is not used for variable-selection decisions.

## 2. Define Full Model and Predictor Blocks
The variable-reduction analysis begins with the eight conceptual predictor blocks included in the broad model:

- `TotalDistance`
- `Coupons`
- `RoundTrip`
- `OnLine`
- `PurchaseWindowGroup`
- `RpCarrier_Model`
- `ItinGeoType`
- `OriginState_Model`

Categorical predictors are treated as complete blocks. If a categorical predictor is removed, all of its associated dummy variables are removed together.

### 2.1 Define Predictor Blocks

In [22]:
predictor_terms = {
    "TotalDistance":
        "TotalDistance",

    "Coupons":
        "Coupons",

    "RoundTrip":
        "C(RoundTrip, Treatment(reference=0))",

    "OnLine":
        "C(OnLine, Treatment(reference=1))",

    "PurchaseWindowGroup":
        "C(PurchaseWindowGroup, Treatment(reference='21AP'))",

    "RpCarrier_Model":
        "C(RpCarrier_Model, Treatment(reference='AA'))",

    "ItinGeoType":
        "C(ItinGeoType, Treatment(reference=2))",

    "OriginState_Model":
        "C(OriginState_Model, Treatment(reference='CA'))"
}

print("Predictor blocks:")
for name in predictor_terms:
    print(f"- {name}")

Predictor blocks:
- TotalDistance
- Coupons
- RoundTrip
- OnLine
- PurchaseWindowGroup
- RpCarrier_Model
- ItinGeoType
- OriginState_Model


### 2.2 Fit the Full Starting Model

In [23]:
full_formula = (
    "TotalAmount ~ "
    + " + ".join(predictor_terms.values())
)

full_model = smf.ols(
    formula=full_formula,
    data=model_df
).fit()

print(f"Observations:        {int(full_model.nobs):,}")
print(f"Model df:            {int(full_model.df_model)}")
print(f"R-squared:           {full_model.rsquared:.4f}")
print(f"Adjusted R-squared:  {full_model.rsquared_adj:.4f}")
print(f"SSE:                 {full_model.ssr:.2f}")
print(f"MSE:                 {full_model.mse_resid:.4f}")
print(f"AIC:                 {full_model.aic:.2f}")
print(f"BIC:                 {full_model.bic:.2f}")

Observations:        45,043
Model df:            60
R-squared:           0.3854
Adjusted R-squared:  0.3846
SSE:                 3434705201.20
MSE:                 76357.3252
AIC:                 634313.98
BIC:                 634845.62


## 3. Backward Selection

Backward elimination provides one nested sequence of models. It starts with all eight predictor blocks and removes the block with the largest non-significant partial-F p-value at each step, stopping when every remaining block is significant at the 5% level.

This stopping rule tests incremental contribution conditional on the blocks currently retained. It does not optimize AIC, BIC, PRESS, or any other criterion across all possible subsets, and its final number of blocks does not bound where a criterion's optimum can occur.

### 3.1 Backward-Selection Procedure

In [24]:
ALPHA = 0.05

current_predictors = list(predictor_terms.keys())
backward_history = []

step = 1

while True:

    current_formula = (
        "TotalAmount ~ "
        + " + ".join(
            predictor_terms[name]
            for name in current_predictors
        )
    )

    current_model = smf.ols(
        formula=current_formula,
        data=model_df
    ).fit()

    removal_tests = []

    for predictor in current_predictors:

        reduced_predictors = [
            name
            for name in current_predictors
            if name != predictor
        ]

        reduced_formula = (
            "TotalAmount ~ "
            + " + ".join(
                predictor_terms[name]
                for name in reduced_predictors
            )
        )

        reduced_model = smf.ols(
            formula=reduced_formula,
            data=model_df
        ).fit()

        f_stat, p_value, df_diff = (
            current_model.compare_f_test(reduced_model)
        )

        removal_tests.append({
            "Step": step,
            "Predictor": predictor,
            "Partial_F": f_stat,
            "p_value": p_value,
            "df_removed": int(df_diff)
        })

    step_table = (
        pd.DataFrame(removal_tests)
        .sort_values("p_value", ascending=False)
        .reset_index(drop=True)
    )

    backward_history.append(step_table)

    print(f"\nBackward-selection step {step}")
    display(step_table)

    worst_predictor = step_table.loc[0, "Predictor"]
    worst_p_value = step_table.loc[0, "p_value"]

    if worst_p_value <= ALPHA:
        break

    print(
        f"Removing {worst_predictor} "
        f"(p = {worst_p_value:.4f})"
    )

    current_predictors.remove(worst_predictor)

    step += 1


Backward-selection step 1


,Step,Predictor,Partial_F,p_value,df_removed
0,1,Coupons,2.815292,9.337688e-02,1
1,1,ItinGeoType,45.415674,1.612706e-11,1
2,1,OnLine,177.807373,1.742369e-40,1
3,1,OriginState_Model,10.078394,4.023863e-64,42
4,1,RoundTrip,1553.964202,0.000000e+00,1
5,1,TotalDistance,5258.104567,0.000000e+00,1
6,1,RpCarrier_Model,430.710030,0.000000e+00,11
7,1,PurchaseWindowGroup,1187.846001,0.000000e+00,2


Removing Coupons (p = 0.0934)

Backward-selection step 2


,Step,Predictor,Partial_F,p_value,df_removed
0,2,ItinGeoType,44.473225,2.608356e-11,1
1,2,OnLine,266.838069,8.251201e-60,1
2,2,OriginState_Model,10.222520,2.654990e-65,42
3,2,RoundTrip,2352.571503,0.000000e+00,1
4,2,TotalDistance,6653.021424,0.000000e+00,1
5,2,RpCarrier_Model,436.842642,0.000000e+00,11
6,2,PurchaseWindowGroup,1189.718462,0.000000e+00,2


In [25]:
selected_predictors = current_predictors.copy()

print("\nFinal predictors retained:")
for predictor in selected_predictors:
    print(f"- {predictor}")

print(
    f"\nNumber of retained predictor blocks: "
    f"{len(selected_predictors)}"
)


Final predictors retained:
- TotalDistance
- RoundTrip
- OnLine
- PurchaseWindowGroup
- RpCarrier_Model
- ItinGeoType
- OriginState_Model

Number of retained predictor blocks: 7


### 3.2 Backward-Selection Result

Backward elimination removes `Coupons` first ($F=2.815$, $p=0.0934$) and then stops because the seven remaining blocks have significant partial F-tests. Its seven-block result is a reference specification for the exhaustive comparison, not an estimated optimal subset size.

In [26]:
backward_history_table = pd.concat(
    backward_history,
    ignore_index=True
)

backward_history_table.to_csv(
    OUTPUT_DIR / "04_backward_selection_history.csv",
    index=False
)

backward_selected = pd.DataFrame({
    "Predictor": selected_predictors
})

backward_selected.to_csv(
    OUTPUT_DIR / "04_backward_selected_predictors.csv",
    index=False
)

## 4. Exhaustive Predictor-Block Subset Search

With eight conceptual predictor blocks, there are $2^8-1=255$ nonempty subsets. Every one is fitted on the same 45,043-row model-building sample. Categorical variables remain complete blocks, so all of a block's dummy columns enter or leave together.

The search compares $R^2$, adjusted $R^2$, Mallows' $C_p$, AIC, BIC/SBC, and exact leave-one-out PRESS. It finds the best subset **within these eight blocks for each criterion**. These criteria can disagree, and selection on the model-building sample does not replace evaluation on the reserved validation sample.

### 4.1 Evaluate All 255 Nonempty Block Subsets

In [ ]:
from scipy.linalg import solve

# Use the full model's design matrix so every subset has the same rows,
# response, categorical coding, and reference levels.
X = full_model.model.exog
y = full_model.model.endog
design_info = full_model.model.data.orig_exog.design_info

block_columns = {
    name: np.arange(
        design_info.term_name_slices[term].start,
        design_info.term_name_slices[term].stop
    )
    for name, term in predictor_terms.items()
}

n = len(y)
full_mse = full_model.mse_resid
tss = np.sum((y - y.mean()) ** 2)
xtx = X.T @ X
xty = X.T @ y
yty = y @ y
predictor_names = list(predictor_terms)
subset_results = []

for r in range(1, len(predictor_names) + 1):
    for subset in combinations(predictor_names, r):
        columns = np.r_[0, *[block_columns[name] for name in subset]]
        x_subset = X[:, columns]
        gram = xtx[np.ix_(columns, columns)]
        beta = solve(gram, xty[columns], assume_a="pos")
        p = len(columns)  # Actual number of parameters, including the intercept
        sse = yty - beta @ xty[columns]

        # h_ii = x_i' (X'X)^-1 x_i; PRESS is exact leave-one-out SSE.
        gram_inverse = solve(gram, np.eye(p), assume_a="pos")
        leverage = np.einsum(
            "ij,jk,ik->i", x_subset, gram_inverse, x_subset,
            optimize=True
        )
        residuals = y - x_subset @ beta
        press = np.sum((residuals / (1 - leverage)) ** 2)

        # These AIC/BIC formulas match statsmodels OLS for Gaussian errors.
        minus_two_log_likelihood = n * (
            np.log(2 * np.pi) + 1 + np.log(sse / n)
        )
        subset_results.append({
            "Predictors": subset,
            "Num_Blocks": len(subset),
            "Num_Parameters": p,
            "R2": 1 - sse / tss,
            "Adj_R2": 1 - (sse / (n - p)) / (tss / (n - 1)),
            "SSE": sse,
            "MSE": sse / (n - p),
            "Cp": sse / full_mse - (n - 2 * p),
            "AIC": minus_two_log_likelihood + 2 * p,
            "BIC": minus_two_log_likelihood + np.log(n) * p,
            "PRESS": press
        })

subset_table = pd.DataFrame(subset_results)
print(f"Number of candidate models: {len(subset_table)}")
display(
    subset_table["Num_Blocks"]
    .value_counts()
    .sort_index()
    .rename("Number of Models")
    .to_frame()
)

### 4.2 Rank All Candidate Models by Selection Criterion

The preferred direction is higher for $R^2$ and adjusted $R^2$, and lower for AIC, BIC/SBC, and PRESS. $R^2$ necessarily favors the full model because adding predictors cannot lower in-sample fit.

Mallows' $C_p$ is used as an adequacy check: a reasonably small value near the actual parameter count $p$ is desirable. Its reference full model has $C_p=p$ by construction, so that equality alone is not evidence that the full model is best. Each criterion is calculated on the model-building sample; PRESS is leave-one-out error within that sample.

In [ ]:
subset_table["Cp_minus_p"] = (
    subset_table["Cp"]
    - subset_table["Num_Parameters"]
)

subset_table["Abs_Cp_minus_p"] = (
    subset_table["Cp_minus_p"].abs()
)

In [ ]:
top_r2 = (
    subset_table
    .sort_values("R2", ascending=False)
    .head(5)
)

top_adj_r2 = (
    subset_table
    .sort_values("Adj_R2", ascending=False)
    .head(5)
)

top_cp = (
    subset_table
    .sort_values(
        ["Abs_Cp_minus_p", "Cp"],
        ascending=[True, True]
    )
    .head(5)
)

top_aic = (
    subset_table
    .sort_values("AIC")
    .head(5)
)

top_bic = (
    subset_table
    .sort_values("BIC")
    .head(5)
)

top_press = (
    subset_table
    .sort_values("PRESS")
    .head(5)
)

In [ ]:
display_cols = [
    "Predictors",
    "Num_Blocks",
    "Num_Parameters",
    "R2",
    "Adj_R2",
    "Cp",
    "Cp_minus_p",
    "AIC",
    "BIC",
    "PRESS"
]

print("Top 5 by R-squared")
display(
    top_r2[display_cols]
    .reset_index(drop=True)
)

print("\nTop 5 by Adjusted R-squared")
display(
    top_adj_r2[display_cols]
    .reset_index(drop=True)
)

print("\nTop 5 by Mallows' Cp")
display(
    top_cp[display_cols]
    .reset_index(drop=True)
)

print("\nTop 5 by AIC")
display(
    top_aic[display_cols]
    .reset_index(drop=True)
)

print("\nTop 5 by BIC / SBC")
display(
    top_bic[display_cols]
    .reset_index(drop=True)
)

print("\nTop 5 by PRESS")
display(
    top_press[display_cols]
    .reset_index(drop=True)
)

In [ ]:
subset_table_export = subset_table.copy()
subset_table_export["Predictors"] = (
    subset_table_export["Predictors"].apply(lambda x: ", ".join(x))
)

subset_table_export.to_csv(
    OUTPUT_DIR / "04_nearby_subset_model_rankings.csv",
    index=False
)

### 4.3 Criterion Winners and Candidate Models

In [ ]:
criterion_winners = pd.DataFrame([
    {
        "Criterion": "R-squared",
        **top_r2.iloc[0].to_dict()
    },
    {
        "Criterion": "Adjusted R-squared",
        **top_adj_r2.iloc[0].to_dict()
    },
    {
        "Criterion": "Mallows' Cp",
        **top_cp.iloc[0].to_dict()
    },
    {
        "Criterion": "AIC",
        **top_aic.iloc[0].to_dict()
    },
    {
        "Criterion": "BIC / SBC",
        **top_bic.iloc[0].to_dict()
    },
    {
        "Criterion": "PRESS",
        **top_press.iloc[0].to_dict()
    }
])

criterion_winners["Predictors"] = (
    criterion_winners["Predictors"]
    .apply(lambda x: ", ".join(x))
)

winner_cols = [
    "Criterion",
    "Predictors",
    "Num_Blocks",
    "Num_Parameters",
    "R2",
    "Adj_R2",
    "Cp",
    "AIC",
    "BIC",
    "PRESS"
]

criterion_winners[winner_cols].to_csv(
    OUTPUT_DIR / "04_criterion_winners.csv",
    index=False
)

display(
    criterion_winners[winner_cols]
)

The exhaustive search gives different winners because the criteria reward fit and complexity differently:

- The full eight-block model leads on $R^2$, adjusted $R^2$, and AIC. Its $C_p=p=61$ is expected because it supplies the reference error variance for $C_p$.
- The seven-block model without `Coupons`, also found by backward elimination, has the lowest PRESS across **all 255** subsets. Its $C_p$ is about 61.82 for $p=60$, close to the reference line; its PRESS is only slightly lower than the full model's.
- BIC/SBC selects a **five-block** model: `TotalDistance`, `RoundTrip`, `OnLine`, `PurchaseWindowGroup`, and `RpCarrier_Model`. This model was outside the original six-to-eight-block search. It has 17 parameters and $C_ppprox410.82$, far above $p=17$, and higher PRESS than the seven- and eight-block models.

The full and seven-block models remain the main explanatory candidates because both retain strong fit and have $C_p$ close to $p$. The five-block model is the global BIC winner in this candidate family and should be reported as a parsimony benchmark. The criteria do not establish one universally optimal model. In particular, the small PRESS difference between the full and seven-block models must be checked on the reserved validation sample before making a predictive claim.

## 5. Variable-Reduction Summary

Backward elimination yields seven blocks, but its stopping point is only one path through the model space. The exhaustive search evaluates all 255 nonempty subsets of the eight predictor blocks, so its reported criterion winners are global **within this defined candidate family and the model-building data**.

Adjusted $R^2$ and AIC favor eight blocks, PRESS favors seven blocks without `Coupons`, and BIC favors five blocks. The full and seven-block specifications are carried forward as the main explanatory candidates; the five-block BIC winner is retained as a documented parsimony benchmark. Final assessment still requires diagnostics and held-out validation.

In [ ]:
candidate_models = pd.DataFrame({
    "Candidate": [
        "Full 8-block model",
        "Backward 7-block model",
        "BIC 5-block benchmark"
    ],
    "Role": [
        "Main explanatory candidate",
        "Main explanatory candidate",
        "Parsimony benchmark"
    ],
    "Predictors": [
        ", ".join(predictor_names),
        ", ".join(selected_predictors),
        ", ".join(top_bic.iloc[0]["Predictors"])
    ]
})

candidate_models.to_csv(
    OUTPUT_DIR / "04_candidate_models.csv",
    index=False
)

display(candidate_models)
